# COINS 02 - OOF Runs And Data Manipulation

This notebook shows the data before final model training. It computes same-family OOF scores, applies Weighted Repair/CWMS, applies Boundary Fill/MSBS, then explicitly previews the final training data that is fed to `model.fit(...)`.


In [1]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np
import pandas as pd
from scipy.stats import norm, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 260)
pd.options.mode.chained_assignment = None

ROOT = Path.cwd()
if not (ROOT / "data").exists() and ROOT.name == "data":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
RAW_RESULTS = ROOT / "docs" / "experiments" / "raw" / "full-benchmark-solution-v2.csv"
SMOKE = os.environ.get("COINS_SMOKE", "0") == "1"

DATA_FILES = {
    "pima": "pima.parquet", "credit-g": "credit-g.parquet", "yeast": "yeast.parquet",
    "ecoli": "ecoli.parquet", "phoneme": "phoneme.parquet", "breast_cancer": "breast_cancer.parquet",
    "ilpd": "ilpd.parquet", "blood": "blood.parquet", "haberman": "haberman.parquet",
    "ionosphere": "ionosphere.parquet", "vehicle_bus": "vehicle_bus.parquet", "glass_float": "glass_float.parquet",
    "abalone": "abalone.parquet", "spambase": "spambase.parquet", "kc1": "kc1.parquet",
}
MINORITY_TEXT = {
    "pima": "tested_positive", "credit-g": "bad", "yeast": "MIT", "ecoli": "im", "phoneme": "nasal",
    "breast_cancer": "malignant", "ilpd": "no_disease", "blood": "donated", "haberman": "died",
    "ionosphere": "bad", "vehicle_bus": "bus", "glass_float": "window_float", "abalone": "rings_gt_10",
    "spambase": "spam", "kc1": "defective",
}
SEEDS_FULL = [13, 17, 23, 29, 31, 37, 41, 43, 47, 53]
PROTOCOLS_FULL = {
    "hidden_minority_low": (0.10, 0.05),
    "hidden_minority_medium": (0.30, 0.10),
    "hidden_minority_high": (0.40, 0.20),
}
GOAL_MINORITY_RATIO = 0.15
BUDGET_RATIO = 0.10
DATA_NAMES = ["pima"] if SMOKE else list(DATA_FILES)
SEEDS = [13] if SMOKE else SEEDS_FULL
PROTOCOLS = {"hidden_minority_medium": PROTOCOLS_FULL["hidden_minority_medium"]} if SMOKE else PROTOCOLS_FULL
print("root", ROOT)
print("data_dir", DATA_DIR)
print("smoke", SMOKE)


def read_data(name: str) -> tuple[pd.DataFrame, np.ndarray, str]:
    df = pd.read_parquet(DATA_DIR / DATA_FILES[name])
    label_col = df.columns[-1]
    y = (df[label_col] == MINORITY_TEXT[name]).astype(int).to_numpy()
    X = df.drop(columns=[label_col])
    return X, y, label_col


def induce_ratio(X: pd.DataFrame, y: np.ndarray, goal_ratio: float, rng: np.random.Generator) -> tuple[pd.DataFrame, np.ndarray]:
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    if len(min_idx) / len(y) <= goal_ratio:
        return X.reset_index(drop=True), y.copy()
    keep_min_n = min(len(min_idx), max(2, int((goal_ratio / (1 - goal_ratio)) * len(maj_idx))))
    keep_min = rng.choice(min_idx, size=keep_min_n, replace=False)
    keep = np.concatenate([maj_idx, keep_min])
    rng.shuffle(keep)
    return X.iloc[keep].reset_index(drop=True), y[keep].copy()


def add_noise(y_clean: np.ndarray, min_to_maj: float, maj_to_min: float, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    y = y_clean.copy()
    mask = np.zeros(len(y), dtype=bool)
    for i, val in enumerate(y_clean):
        p = min_to_maj if val == 1 else maj_to_min
        if rng.random() < p:
            y[i] = 1 - y[i]
            mask[i] = True
    return y, mask


def preprocessor_for(X: pd.DataFrame) -> ColumnTransformer:
    cat = list(X.select_dtypes(include=["object", "category", "bool"]).columns)
    num = [c for c in X.columns if c not in cat]
    try:
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        enc = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", enc)]), cat),
    ])


def split_data(name: str, seed: int, min_to_maj: float, maj_to_min: float) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame, list[str]]:
    rng = np.random.default_rng(seed)
    X_raw, y_raw, _label_col = read_data(name)
    X_tr_df, X_te_df, y_tr_clean, y_te = train_test_split(X_raw, y_raw, test_size=0.25, stratify=y_raw, random_state=seed)
    X_tr_df, y_tr_clean = induce_ratio(X_tr_df.reset_index(drop=True), y_tr_clean, GOAL_MINORITY_RATIO, rng)
    prep = preprocessor_for(X_tr_df)
    X_tr = prep.fit_transform(X_tr_df).astype(float)
    X_te = prep.transform(X_te_df.reset_index(drop=True)).astype(float)
    y_noisy, noisy_mask = add_noise(y_tr_clean, min_to_maj, maj_to_min, rng)
    try:
        feature_names = prep.get_feature_names_out().tolist()
    except Exception:
        feature_names = [f"feature_{i}" for i in range(X_tr.shape[1])]
    return X_tr, X_te, y_tr_clean, y_noisy, y_te, noisy_mask, X_tr_df, feature_names


def make_model(name: str, seed: int, balanced: bool = False) -> LogisticRegression | SVC:
    if name == "lr":
        return LogisticRegression(class_weight="balanced" if balanced else None, max_iter=1000, random_state=seed)
    if name == "svm":
        return SVC(kernel="rbf", probability=True, class_weight="balanced" if balanced else None, random_state=seed)
    raise ValueError(name)


def min_scores(model: LogisticRegression | SVC, X: np.ndarray) -> np.ndarray:
    return model.predict_proba(X)[:, list(model.classes_).index(1)]


def same_family_oof(X: np.ndarray, y: np.ndarray, model_name: str, seed: int) -> np.ndarray:
    out = np.full(len(y), np.nan)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in cv.split(X, y):
        model = make_model(model_name, seed, balanced=True)
        model.fit(X[tr], y[tr])
        out[va] = min_scores(model, X[va])
    out[y == 1] = np.nan
    return out


def msbs(X: np.ndarray, y: np.ndarray, scores: np.ndarray, budget: int, seed: int) -> tuple[np.ndarray, np.ndarray, int]:
    rng = np.random.default_rng(seed)
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    if len(min_idx) == 0 or len(maj_idx) == 0 or budget <= 0:
        return X.copy(), y.copy(), 0
    vals = np.nan_to_num(scores[maj_idx], nan=0.0)
    probs = np.full(len(maj_idx), 1 / len(maj_idx)) if vals.sum() <= 0 else vals / vals.sum()
    synth = []
    for _ in range(budget):
        anchor = X[rng.choice(min_idx)]
        seed_point = X[rng.choice(maj_idx, p=probs)]
        synth.append(anchor + rng.random() * (seed_point - anchor))
    return np.vstack([X, np.asarray(synth)]), np.concatenate([y, np.ones(len(synth), dtype=int)]), len(synth)


def build_coins_training_data(X: np.ndarray, y_noisy: np.ndarray, scores: np.ndarray, budget: int, seed: int) -> dict[str, Any]:
    weights = np.ones(len(y_noisy))
    majority = y_noisy == 0
    weights[majority] = 1 - np.nan_to_num(scores[majority], nan=0.0)
    X_aug, y_aug, n_synth = msbs(X, y_noisy, scores, budget, seed)
    final_weights = np.concatenate([np.clip(weights, 0, 1), np.ones(n_synth)])
    row_stage = np.array(["original_weighted_repair"] * len(y_noisy) + ["synthetic_boundary_fill"] * n_synth)
    return {"X_aug": X_aug, "y_aug": y_aug, "weights": final_weights, "original_weights": weights, "row_stage": row_stage, "n_synthetic": n_synth}


def preview_frame(X: np.ndarray, y: np.ndarray, weights: np.ndarray, row_stage: np.ndarray, feature_names: list[str], dataset: str, model: str, seed: int, protocol: str, start: int = 0, n: int = 5) -> pd.DataFrame:
    end = min(start + n, len(y))
    cols = feature_names[: min(6, X.shape[1])]
    df = pd.DataFrame(X[start:end, : len(cols)], columns=cols)
    df.insert(0, "dataset", dataset)
    df.insert(1, "model", model)
    df.insert(2, "seed", seed)
    df.insert(3, "noise_protocol", protocol)
    df.insert(4, "row_index", np.arange(start, end))
    df.insert(5, "observed_label_for_training", y[start:end])
    df.insert(6, "sample_weight", weights[start:end])
    df.insert(7, "row_stage", row_stage[start:end])
    return df


root /home/than-minh/project/DSP
data_dir /home/than-minh/project/DSP/data
smoke False


## Noise Protocol Explanation

A noise protocol is controlled label corruption on training labels only. `hidden_minority_medium`, for example, flips 30% of true minority training labels into majority labels and 10% of true majority labels into minority labels. Test labels stay clean.


In [2]:
protocol_table = pd.DataFrame([
    {"noise_protocol": "hidden_minority_low", "minority_to_majority_flip_rate": 0.10, "majority_to_minority_flip_rate": 0.05},
    {"noise_protocol": "hidden_minority_medium", "minority_to_majority_flip_rate": 0.30, "majority_to_minority_flip_rate": 0.10},
    {"noise_protocol": "hidden_minority_high", "minority_to_majority_flip_rate": 0.40, "majority_to_minority_flip_rate": 0.20},
])
display(protocol_table)


,noise_protocol,minority_to_majority_flip_rate,majority_to_minority_flip_rate
0,hidden_minority_low,0.1,0.05
1,hidden_minority_medium,0.3,0.10
2,hidden_minority_high,0.4,0.20


## Full OOF Score Runs

OOF means each row is scored only by a model that did not train on that row. Same-family means LR uses balanced LR for scoring and SVM uses balanced SVM for scoring.


In [3]:
OOF_MODELS = ["lr", "svm"]
oof_cache = {}
oof_rows = []
combo_total = len(DATA_NAMES) * len(OOF_MODELS) * len(SEEDS) * len(PROTOCOLS)
start = time.time()
combo_i = 0
for dataset in DATA_NAMES:
    for model in OOF_MODELS:
        for seed in SEEDS:
            for protocol, (mn, mj) in PROTOCOLS.items():
                combo_i += 1
                if combo_i <= 5 or combo_i % 100 == 0 or combo_i == combo_total:
                    print(f"[OOF {combo_i}/{combo_total}] {dataset} {model} seed={seed} {protocol}")
                X, _X_te, y_clean, y_noisy, _y_te, mask, _X_df, _feature_names = split_data(dataset, seed, mn, mj)
                scores = same_family_oof(X, y_noisy, model, seed)
                oof_cache[(dataset, model, seed, protocol)] = scores
                observed_majority = y_noisy == 0
                hidden_minority = (y_clean == 1) & observed_majority
                clean_majority = (y_clean == 0) & observed_majority
                target = hidden_minority[observed_majority].astype(int)
                vals = np.nan_to_num(scores[observed_majority], nan=0.0)
                ap = np.nan if target.sum() == 0 or target.sum() == len(target) else float(average_precision_score(target, vals))
                hidden_mean = float(np.nanmean(scores[hidden_minority])) if hidden_minority.any() else np.nan
                clean_mean = float(np.nanmean(scores[clean_majority])) if clean_majority.any() else np.nan
                oof_rows.append({
                    "dataset": dataset,
                    "model": model,
                    "seed": seed,
                    "noise_protocol": protocol,
                    "train_rows": len(y_noisy),
                    "observed_majority_rows": int(observed_majority.sum()),
                    "observed_minority_rows": int((y_noisy == 1).sum()),
                    "hidden_minority_rows": int(hidden_minority.sum()),
                    "true_noise_rate": float(mask.mean()),
                    "majority_score_mean": float(np.nanmean(scores[observed_majority])),
                    "hidden_minority_score_mean": hidden_mean,
                    "clean_majority_score_mean": clean_mean,
                    "score_gap": hidden_mean - clean_mean if np.isfinite(hidden_mean) and np.isfinite(clean_mean) else np.nan,
                    "hidden_minority_ap": ap,
                })
oof_table = pd.DataFrame(oof_rows)
print("OOF rows", len(oof_table), "minutes", round((time.time() - start) / 60, 2))
display(oof_table.head(20))


[OOF 1/900] pima lr seed=13 hidden_minority_low
[OOF 2/900] pima lr seed=13 hidden_minority_medium
[OOF 3/900] pima lr seed=13 hidden_minority_high
[OOF 4/900] pima lr seed=17 hidden_minority_low
[OOF 5/900] pima lr seed=17 hidden_minority_medium


[OOF 100/900] credit-g svm seed=29 hidden_minority_low


[OOF 200/900] ecoli lr seed=41 hidden_minority_medium


[OOF 300/900] phoneme svm seed=53 hidden_minority_high


[OOF 400/900] ilpd svm seed=29 hidden_minority_low


...

[OOF 700/900] glass_float svm seed=29 hidden_minority_low


[OOF 800/900] spambase lr seed=41 hidden_minority_medium


[OOF 900/900] kc1 svm seed=53 hidden_minority_high


OOF rows 900 minutes 4.53


,dataset,model,seed,noise_protocol,train_rows,observed_majority_rows,observed_minority_rows,hidden_minority_rows,true_noise_rate,majority_score_mean,hidden_minority_score_mean,clean_majority_score_mean,score_gap,hidden_minority_ap
0,pima,lr,13,hidden_minority_low,441,361,80,9,0.072562,0.434049,0.671954,0.427966,0.243987,0.299003
1,pima,lr,13,hidden_minority_medium,441,355,86,20,0.136054,0.443601,0.606101,0.433900,0.172201,0.243617
2,pima,lr,13,hidden_minority_high,441,326,115,26,0.229025,0.490764,0.549048,0.485712,0.063335,0.244196
3,pima,lr,17,hidden_minority_low,441,366,75,5,0.043084,0.417723,0.683501,0.414042,0.269459,0.248102
4,pima,lr,17,hidden_minority_medium,441,357,84,15,0.108844,0.459733,0.632909,0.452138,0.180771,0.229675
5,pima,lr,17,hidden_minority_high,441,319,122,25,0.240363,0.487500,0.578301,0.479778,0.098523,0.217560
6,pima,lr,23,hidden_minority_low,441,366,75,5,0.043084,0.415379,0.562093,0.413347,0.148746,0.044150
7,pima,lr,23,hidden_minority_medium,441,352,89,15,0.120181,0.457508,0.592408,0.451503,0.140905,0.120571
8,pima,lr,23,hidden_minority_high,441,325,116,24,0.222222,0.489491,0.552767,0.484445,0.068322,0.223410
9,pima,lr,29,hidden_minority_low,441,369,72,9,0.054422,0.351702,0.568588,0.346280,0.222308,0.081805


## OOF Result Tables


In [4]:
display(oof_table.groupby(["model", "noise_protocol"])[["hidden_minority_rows", "majority_score_mean", "hidden_minority_score_mean", "clean_majority_score_mean", "score_gap", "hidden_minority_ap"]].mean().reset_index())
display(oof_table.groupby(["dataset", "model"])[["score_gap", "hidden_minority_ap", "true_noise_rate"]].mean().reset_index().sort_values(["model", "dataset"]))


,model,noise_protocol,hidden_minority_rows,majority_score_mean,hidden_minority_score_mean,clean_majority_score_mean,score_gap,hidden_minority_ap
0,lr,hidden_minority_high,52.580000,0.469391,0.588150,0.459482,0.128668,0.323167
1,lr,hidden_minority_low,12.886667,0.364602,0.678763,0.359451,0.318582,0.276328
2,lr,hidden_minority_medium,39.100000,0.425017,0.628001,0.413728,0.214273,0.330423
3,svm,hidden_minority_high,52.580000,0.252832,0.308838,0.248252,0.060586,0.278637
4,svm,hidden_minority_low,12.886667,0.145820,0.406597,0.141563,0.265205,0.280308
5,svm,hidden_minority_medium,39.100000,0.178053,0.315379,0.170562,0.144817,0.334353


,dataset,model,score_gap,hidden_minority_ap,true_noise_rate
0,abalone,lr,0.191104,0.257909,0.137153
2,blood,lr,0.085214,0.163191,0.138419
4,breast_cancer,lr,0.497621,0.761593,0.138535
6,credit-g,lr,0.126297,0.109079,0.134954
8,ecoli,lr,0.332783,0.506121,0.130994
10,glass_float,lr,0.116273,0.106189,0.135433
12,haberman,lr,0.058295,0.127941,0.131980
14,ilpd,lr,0.080801,0.090787,0.138601
16,ionosphere,lr,0.344099,0.543658,0.132492
18,kc1,lr,0.153101,0.204774,0.138020


## Weighted Repair And Boundary Fill Checkpoints

For each dataset, this table shows row counts before COINS, after Weighted Repair/CWMS, and after Boundary Fill/MSBS. The final checkpoint is the training matrix used by COINS.


In [5]:
checkpoint_rows = []
preview_before = []
preview_weighted = []
preview_final_original_head = []
preview_final_synthetic_head = []
for dataset in DATA_NAMES:
    for model in ["lr", "svm"]:
        seed = SEEDS_FULL[0]
        protocol = "hidden_minority_medium"
        mn, mj = PROTOCOLS_FULL[protocol]
        X, _X_te, y_clean, y_noisy, _y_te, mask, _X_df, feature_names = split_data(dataset, seed, mn, mj)
        scores = oof_cache.get((dataset, model, seed, protocol))
        if scores is None:
            scores = same_family_oof(X, y_noisy, model, seed)
            oof_cache[(dataset, model, seed, protocol)] = scores
        budget = max(1, int(round(BUDGET_RATIO * len(y_noisy))))
        coins_data = build_coins_training_data(X, y_noisy, scores, budget, seed)
        observed_majority = y_noisy == 0
        threshold = np.nanquantile(scores[observed_majority], 0.80)
        suspicious = observed_majority & (np.nan_to_num(scores, nan=0.0) >= threshold)
        checkpoint_rows.extend([
            {"dataset": dataset, "model": model, "checkpoint": "before_coins", "rows": len(y_noisy), "minority_rows": int((y_noisy == 1).sum()), "majority_rows": int((y_noisy == 0).sum()), "synthetic_rows": 0, "mean_weight": 1.0, "fit_ready": False},
            {"dataset": dataset, "model": model, "checkpoint": "after_weighted_repair_cwms", "rows": len(y_noisy), "minority_rows": int((y_noisy == 1).sum()), "majority_rows": int((y_noisy == 0).sum()), "synthetic_rows": 0, "mean_weight": float(coins_data["original_weights"].mean()), "fit_ready": False},
            {"dataset": dataset, "model": model, "checkpoint": "after_boundary_fill_msbs_final_training_data", "rows": len(coins_data["y_aug"]), "minority_rows": int((coins_data["y_aug"] == 1).sum()), "majority_rows": int((coins_data["y_aug"] == 0).sum()), "synthetic_rows": int(coins_data["n_synthetic"]), "mean_weight": float(coins_data["weights"].mean()), "fit_ready": True},
        ])
        before_stage = np.array(["before_coins"] * len(y_noisy))
        weighted_stage = np.array(["weighted_repair_cwms"] * len(y_noisy))
        preview_before.append(preview_frame(X, y_noisy, np.ones(len(y_noisy)), before_stage, feature_names, dataset, model, seed, protocol))
        preview_weighted.append(preview_frame(X, y_noisy, coins_data["original_weights"], weighted_stage, feature_names, dataset, model, seed, protocol))
        preview_final_original_head.append(preview_frame(coins_data["X_aug"], coins_data["y_aug"], coins_data["weights"], coins_data["row_stage"], feature_names, dataset, model, seed, protocol, start=0))
        synth_start = len(y_noisy)
        preview_final_synthetic_head.append(preview_frame(coins_data["X_aug"], coins_data["y_aug"], coins_data["weights"], coins_data["row_stage"], feature_names, dataset, model, seed, protocol, start=synth_start))

checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)


,dataset,model,checkpoint,rows,minority_rows,majority_rows,synthetic_rows,mean_weight,fit_ready
0,pima,lr,before_coins,441,86,355,0,1.000000,False
1,pima,lr,after_weighted_repair_cwms,441,86,355,0,0.642906,False
2,pima,lr,after_boundary_fill_msbs_final_training_data,485,130,355,44,0.675302,True
3,pima,svm,before_coins,441,86,355,0,1.000000,False
4,pima,svm,after_weighted_repair_cwms,441,86,355,0,0.844398,False
5,pima,svm,after_boundary_fill_msbs_final_training_data,485,130,355,44,0.858515,True
6,credit-g,lr,before_coins,617,120,497,0,1.000000,False
7,credit-g,lr,after_weighted_repair_cwms,617,120,497,0,0.650922,False
8,credit-g,lr,after_boundary_fill_msbs_final_training_data,679,182,497,62,0.682797,True
9,credit-g,svm,before_coins,617,120,497,0,1.000000,False


## Before COINS `head(5)` For Each Dataset And Model


In [6]:
for df in preview_before:
    print("DATASET", df["dataset"].iloc[0], "MODEL", df["model"].iloc[0])
    display(df.head(5))


DATASET pima MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,lr,13,hidden_minority_medium,0,0,1.0,before_coins,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,lr,13,hidden_minority_medium,1,0,1.0,before_coins,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,lr,13,hidden_minority_medium,2,0,1.0,before_coins,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,lr,13,hidden_minority_medium,3,0,1.0,before_coins,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,lr,13,hidden_minority_medium,4,0,1.0,before_coins,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET pima MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,svm,13,hidden_minority_medium,0,0,1.0,before_coins,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,svm,13,hidden_minority_medium,1,0,1.0,before_coins,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,svm,13,hidden_minority_medium,2,0,1.0,before_coins,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,svm,13,hidden_minority_medium,3,0,1.0,before_coins,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,svm,13,hidden_minority_medium,4,0,1.0,before_coins,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET credit-g MODEL lr


...

,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__word_freq_make,num__word_freq_address,num__word_freq_all,num__word_freq_3d,num__word_freq_our,num__word_freq_over
0,spambase,svm,13,hidden_minority_medium,0,0,1.0,before_coins,1.415177,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
1,spambase,svm,13,hidden_minority_medium,1,0,1.0,before_coins,0.613899,-0.155892,0.106608,-0.031669,-0.350292,-0.290500
2,spambase,svm,13,hidden_minority_medium,2,1,1.0,before_coins,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
3,spambase,svm,13,hidden_minority_medium,3,0,1.0,before_coins,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
4,spambase,svm,13,hidden_minority_medium,4,0,1.0,before_coins,1.230266,-0.155892,-0.463470,-0.031669,0.382144,1.902169


DATASET kc1 MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,lr,13,hidden_minority_medium,0,0,1.0,before_coins,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,lr,13,hidden_minority_medium,1,0,1.0,before_coins,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,lr,13,hidden_minority_medium,2,0,1.0,before_coins,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,lr,13,hidden_minority_medium,3,1,1.0,before_coins,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,lr,13,hidden_minority_medium,4,0,1.0,before_coins,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


DATASET kc1 MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,svm,13,hidden_minority_medium,0,0,1.0,before_coins,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,svm,13,hidden_minority_medium,1,0,1.0,before_coins,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,svm,13,hidden_minority_medium,2,0,1.0,before_coins,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,svm,13,hidden_minority_medium,3,1,1.0,before_coins,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,svm,13,hidden_minority_medium,4,0,1.0,before_coins,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


## After Weighted Repair / CWMS `head(5)` For Each Dataset And Model


In [7]:
for df in preview_weighted:
    print("DATASET", df["dataset"].iloc[0], "MODEL", df["model"].iloc[0])
    display(df.head(5))


DATASET pima MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,lr,13,hidden_minority_medium,0,0,0.378133,weighted_repair_cwms,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,lr,13,hidden_minority_medium,1,0,0.413516,weighted_repair_cwms,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,lr,13,hidden_minority_medium,2,0,0.407984,weighted_repair_cwms,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,lr,13,hidden_minority_medium,3,0,0.763640,weighted_repair_cwms,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,lr,13,hidden_minority_medium,4,0,0.453327,weighted_repair_cwms,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET pima MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,svm,13,hidden_minority_medium,0,0,0.809580,weighted_repair_cwms,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,svm,13,hidden_minority_medium,1,0,0.889498,weighted_repair_cwms,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,svm,13,hidden_minority_medium,2,0,0.736508,weighted_repair_cwms,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,svm,13,hidden_minority_medium,3,0,0.829724,weighted_repair_cwms,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,svm,13,hidden_minority_medium,4,0,0.849022,weighted_repair_cwms,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET credit-g MODEL lr


...

,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__word_freq_make,num__word_freq_address,num__word_freq_all,num__word_freq_3d,num__word_freq_our,num__word_freq_over
0,spambase,svm,13,hidden_minority_medium,0,0,0.811851,weighted_repair_cwms,1.415177,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
1,spambase,svm,13,hidden_minority_medium,1,0,0.892791,weighted_repair_cwms,0.613899,-0.155892,0.106608,-0.031669,-0.350292,-0.290500
2,spambase,svm,13,hidden_minority_medium,2,1,1.000000,weighted_repair_cwms,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
3,spambase,svm,13,hidden_minority_medium,3,0,0.916250,weighted_repair_cwms,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
4,spambase,svm,13,hidden_minority_medium,4,0,0.903798,weighted_repair_cwms,1.230266,-0.155892,-0.463470,-0.031669,0.382144,1.902169


DATASET kc1 MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,lr,13,hidden_minority_medium,0,0,0.402631,weighted_repair_cwms,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,lr,13,hidden_minority_medium,1,0,0.608840,weighted_repair_cwms,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,lr,13,hidden_minority_medium,2,0,0.445835,weighted_repair_cwms,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,lr,13,hidden_minority_medium,3,1,1.000000,weighted_repair_cwms,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,lr,13,hidden_minority_medium,4,0,0.545452,weighted_repair_cwms,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


DATASET kc1 MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,svm,13,hidden_minority_medium,0,0,0.773460,weighted_repair_cwms,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,svm,13,hidden_minority_medium,1,0,0.847363,weighted_repair_cwms,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,svm,13,hidden_minority_medium,2,0,0.706204,weighted_repair_cwms,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,svm,13,hidden_minority_medium,3,1,1.000000,weighted_repair_cwms,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,svm,13,hidden_minority_medium,4,0,0.848468,weighted_repair_cwms,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


## Final Training Data Fed To Model: Original Weighted Rows `head(5)`

This is the actual final `X_aug`, `y_aug`, and `sample_weight` input after Boundary Fill. These first five rows are original rows with COINS weights.


In [8]:
for df in preview_final_original_head:
    print("DATASET", df["dataset"].iloc[0], "MODEL", df["model"].iloc[0])
    display(df.head(5))


DATASET pima MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,lr,13,hidden_minority_medium,0,0,0.378133,original_weighted_repair,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,lr,13,hidden_minority_medium,1,0,0.413516,original_weighted_repair,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,lr,13,hidden_minority_medium,2,0,0.407984,original_weighted_repair,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,lr,13,hidden_minority_medium,3,0,0.763640,original_weighted_repair,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,lr,13,hidden_minority_medium,4,0,0.453327,original_weighted_repair,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET pima MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,svm,13,hidden_minority_medium,0,0,0.809580,original_weighted_repair,0.501533,0.659328,0.637019,-1.273588,-0.698213,-0.583918
1,pima,svm,13,hidden_minority_medium,1,0,0.889498,original_weighted_repair,0.823477,0.351315,0.201982,1.677644,1.709804,0.355337
2,pima,svm,13,hidden_minority_medium,2,0,0.736508,original_weighted_repair,0.501533,1.172681,0.365121,-1.273588,-0.698213,-0.155728
3,pima,svm,13,hidden_minority_medium,3,0,0.829724,original_weighted_repair,-1.108190,0.043303,0.637019,0.956232,2.285634,1.819470
4,pima,svm,13,hidden_minority_medium,4,0,0.849022,original_weighted_repair,-1.108190,1.172681,0.908918,2.267891,-0.698213,1.626094


DATASET credit-g MODEL lr


...

,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__word_freq_make,num__word_freq_address,num__word_freq_all,num__word_freq_3d,num__word_freq_our,num__word_freq_over
0,spambase,svm,13,hidden_minority_medium,0,0,0.811851,original_weighted_repair,1.415177,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
1,spambase,svm,13,hidden_minority_medium,1,0,0.892791,original_weighted_repair,0.613899,-0.155892,0.106608,-0.031669,-0.350292,-0.290500
2,spambase,svm,13,hidden_minority_medium,2,1,1.000000,original_weighted_repair,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
3,spambase,svm,13,hidden_minority_medium,3,0,0.916250,original_weighted_repair,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
4,spambase,svm,13,hidden_minority_medium,4,0,0.903798,original_weighted_repair,1.230266,-0.155892,-0.463470,-0.031669,0.382144,1.902169


DATASET kc1 MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,lr,13,hidden_minority_medium,0,0,0.402631,original_weighted_repair,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,lr,13,hidden_minority_medium,1,0,0.608840,original_weighted_repair,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,lr,13,hidden_minority_medium,2,0,0.445835,original_weighted_repair,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,lr,13,hidden_minority_medium,3,1,1.000000,original_weighted_repair,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,lr,13,hidden_minority_medium,4,0,0.545452,original_weighted_repair,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


DATASET kc1 MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,svm,13,hidden_minority_medium,0,0,0.773460,original_weighted_repair,4.372250,5.688859,7.945143,6.571513,4.377983,4.467136
1,kc1,svm,13,hidden_minority_medium,1,0,0.847363,original_weighted_repair,-0.620252,-0.473054,-0.316316,-0.459450,-0.565969,-0.508153
2,kc1,svm,13,hidden_minority_medium,2,0,0.706204,original_weighted_repair,0.704290,0.553932,0.601624,0.712377,0.617093,0.481918
3,kc1,svm,13,hidden_minority_medium,3,1,1.000000,original_weighted_repair,-0.280626,-0.473054,-0.316316,-0.459450,-0.167464,-0.235943
4,kc1,svm,13,hidden_minority_medium,4,0,0.848468,original_weighted_repair,-0.450439,-0.473054,-0.316316,-0.459450,-0.441436,-0.421575


## Final Training Data Fed To Model: Synthetic Boundary Fill Rows `head(5)`

These are the appended synthetic minority rows. Showing them separately makes the actual Boundary Fill additions visible instead of hiding them after the original rows.


In [9]:
for df in preview_final_synthetic_head:
    print("DATASET", df["dataset"].iloc[0], "MODEL", df["model"].iloc[0])
    display(df.head(5))


DATASET

 pima MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,lr,13,hidden_minority_medium,441,1,1.0,synthetic_boundary_fill,0.719542,0.964914,0.497430,0.553955,0.371668,-0.342471
1,pima,lr,13,hidden_minority_medium,442,1,1.0,synthetic_boundary_fill,1.417658,1.472890,-0.316627,0.431568,4.219960,0.406322
2,pima,lr,13,hidden_minority_medium,443,1,1.0,synthetic_boundary_fill,1.147115,0.041052,-0.124869,-1.273588,-0.698213,0.241094
3,pima,lr,13,hidden_minority_medium,444,1,1.0,synthetic_boundary_fill,-0.425161,0.272467,0.106444,0.830049,0.281288,1.128672
4,pima,lr,13,hidden_minority_medium,445,1,1.0,synthetic_boundary_fill,-0.949005,0.078287,-3.489291,-1.111451,-0.642995,0.324911


DATASET pima MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__preg,num__plas,num__pres,num__skin,num__insu,num__mass
0,pima,svm,13,hidden_minority_medium,441,1,1.0,synthetic_boundary_fill,0.719542,0.964914,0.497430,0.553955,0.371668,-0.342471
1,pima,svm,13,hidden_minority_medium,442,1,1.0,synthetic_boundary_fill,1.318242,1.250959,-0.299834,0.375876,4.145601,0.369001
2,pima,svm,13,hidden_minority_medium,443,1,1.0,synthetic_boundary_fill,1.141187,0.041863,-0.125155,-1.270655,-0.694908,0.245527
3,pima,svm,13,hidden_minority_medium,444,1,1.0,synthetic_boundary_fill,0.208943,-0.940859,-0.536195,0.184186,-0.192997,-0.408429
4,pima,svm,13,hidden_minority_medium,445,1,1.0,synthetic_boundary_fill,-1.108190,0.021881,-3.713356,-1.273588,-0.698213,0.238402


...

,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__word_freq_make,num__word_freq_address,num__word_freq_all,num__word_freq_3d,num__word_freq_our,num__word_freq_over
0,spambase,svm,13,hidden_minority_medium,2460,1,1.0,synthetic_boundary_fill,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
1,spambase,svm,13,hidden_minority_medium,2461,1,1.0,synthetic_boundary_fill,-0.279833,-0.155892,-0.351169,-0.031669,-0.350292,-0.290500
2,spambase,svm,13,hidden_minority_medium,2462,1,1.0,synthetic_boundary_fill,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
3,spambase,svm,13,hidden_minority_medium,2463,1,1.0,synthetic_boundary_fill,-0.279833,-0.155892,-0.463470,-0.031669,-0.350292,-0.290500
4,spambase,svm,13,hidden_minority_medium,2464,1,1.0,synthetic_boundary_fill,-0.264595,-0.155892,-0.398671,-0.031669,-0.334279,-0.242561


DATASET kc1 MODEL lr


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,lr,13,hidden_minority_medium,1572,1,1.0,synthetic_boundary_fill,-0.011866,-0.278977,-0.142846,-0.238002,-0.375738,-0.323338
1,kc1,lr,13,hidden_minority_medium,1573,1,1.0,synthetic_boundary_fill,-0.481291,-0.393771,-0.316316,-0.368986,-0.444879,-0.406699
2,kc1,lr,13,hidden_minority_medium,1574,1,1.0,synthetic_boundary_fill,1.446195,1.319443,1.514734,1.585853,1.036442,0.970528
3,kc1,lr,13,hidden_minority_medium,1575,1,1.0,synthetic_boundary_fill,-0.587220,-0.161684,-0.246568,-0.108618,-0.456391,-0.419852
4,kc1,lr,13,hidden_minority_medium,1576,1,1.0,synthetic_boundary_fill,-0.399306,-0.473054,-0.316316,-0.459450,-0.260518,-0.322639


DATASET kc1 MODEL svm


,dataset,model,seed,noise_protocol,row_index,observed_label_for_training,sample_weight,row_stage,num__loc,num__v(g),num__ev(g),num__iv(g),num__n,num__v
0,kc1,svm,13,hidden_minority_medium,1572,1,1.0,synthetic_boundary_fill,-0.369944,-0.278977,-0.142846,-0.238002,-0.335338,-0.310024
1,kc1,svm,13,hidden_minority_medium,1573,1,1.0,synthetic_boundary_fill,-0.481291,-0.393771,-0.316316,-0.368986,-0.444879,-0.406699
2,kc1,svm,13,hidden_minority_medium,1574,1,1.0,synthetic_boundary_fill,1.446195,1.319443,1.514734,1.585853,1.036442,0.970528
3,kc1,svm,13,hidden_minority_medium,1575,1,1.0,synthetic_boundary_fill,0.549960,1.608229,3.369393,-0.108618,1.456798,1.356845
4,kc1,svm,13,hidden_minority_medium,1576,1,1.0,synthetic_boundary_fill,-0.298550,-0.409580,-0.316316,-0.387024,-0.158919,-0.239659
